# Adaptive Multi-Agent System for Cryptocurrency Mining + Process Mining Event Log

This notebook implements an **adaptive multi-agent system (MAS)** in a simulated cryptocurrency mining environment.

The important modelling shift is that agents are **not assigned fixed strategies** such as *aggressive* or *conservative*. Instead, every agent has action preferences that are updated from experience.

The model contains:

- heterogeneous autonomous mining agents;
- stochastic coin-price movement;
- local energy grids with random price shocks;
- weighted-lottery mining rewards;
- emergent pool joining/leaving;
- simple reinforcement-style learning;
- social imitation of successful agents;
- an event log for process mining.

Conceptually:

```text
Simulated uncertain environment + adaptive autonomous agents
        ↓
Emergent behaviour: invest, pause, pool, exit, imitate, survive
        ↓
Event log
        ↓
Process mining analysis
```

The environment is simulated, but the **agent behaviour is adaptive and decentralized**.

## 1. Installation notes

Run the code cell below to directly install all required Python libraries for the multi-agent simulation, visualization, and process mining analysis (`mesa`, `pandas`, `numpy`, `matplotlib`, `seaborn`, and `pm4py`).

In [ ]:
# Install required libraries directly within the notebook
%pip install mesa pandas numpy matplotlib seaborn pm4py

## 2. Imports

Import necessary libraries for multi-agent modeling (`mesa`), data manipulation (`pandas`, `numpy`), visualization (`matplotlib`), and process mining (`pm4py`).

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import mesa
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Mesa version:", mesa.__version__)

## 3. Helper functions

This section defines utility functions used across the simulation:
- **Gini coefficient (`gini`)**: Measures wealth concentration among agents over time (0 = perfect equality, 1 = maximum inequality).
- **Softmax action selection (`softmax_choice`)**: Converts learned action Q-values into a probabilistic choice distribution, allowing bounded-rational exploration.

In [ ]:
def gini(values) -> float:
    """Return the Gini coefficient for a list/array of non-negative values."""
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return 0.0
    arr = arr - arr.min()
    if arr.sum() == 0:
        return 0.0
    arr = np.sort(arr)
    n = len(arr)
    index = np.arange(1, n + 1)
    return float((2 * np.sum(index * arr) / (n * np.sum(arr))) - ((n + 1) / n))


def softmax_choice(scores: Dict[str, float], rng: np.random.Generator, temperature: float = 1.0) -> str:
    """Sample an action from a softmax distribution over action scores."""
    actions = list(scores.keys())
    raw = np.array([scores[a] for a in actions], dtype=float)
    raw = raw - raw.max()
    temp = max(temperature, 1e-6)
    probs = np.exp(raw / temp)
    probs = probs / probs.sum()
    return str(rng.choice(actions, p=probs))

## 4. Model parameters

The degree of emergent behavior in the simulation is governed by the environment and agent parameters defined below.

Important parameters:
- `learning_rate`: how fast agents update action preferences.
- `exploration_rate`: probability that agents try a non-obvious action.
- `imitation_rate`: probability that agents copy a successful peer.
- `price_volatility`: how unstable the coin price is.
- `energy_shock_prob`: probability that a local grid receives an energy price shock.
- `pool_fee`: fee paid when a pool wins.

In [ ]:
@dataclass
class MiningParams:
    n_agents: int = 500
    cycles: int = 200
    seed: int = 42

    # Market / mining environment
    initial_coin_price: float = 1_000.0
    block_reward_coins: float = 1.0
    price_drift: float = 0.0005
    price_volatility: float = 0.055
    hardware_failure_prob: float = 0.015
    energy_shock_prob: float = 0.04
    energy_shock_multiplier_min: float = 1.5
    energy_shock_multiplier_max: float = 3.0

    # Cost and hardware scaling
    hash_purchase_cost: float = 8.0
    resale_value_fraction: float = 0.45
    cost_per_hash_factor: float = 0.025
    repair_cost_per_hash: float = 3.0
    min_hash_power: float = 1.0

    # Agent learning / bounded rationality
    learning_rate: float = 0.18
    exploration_rate: float = 0.08
    imitation_rate: float = 0.03
    irrationality_rate: float = 0.01
    decision_temperature: float = 0.85
    max_recent_profit_memory: int = 12

    # Pooling
    max_pools: int = 30
    pool_fee: float = 0.02

    # Exit rules
    bankruptcy_threshold: float = 20.0
    # Capitulation: rolling loss over the recent-profit window, as a
    # fraction of initial capital. Replaces winless-cycle counting.
    loss_tolerance_min: float = 0.02
    loss_tolerance_max: float = 0.12

## 5. Adaptive mining agent

Each agent has capital, hash power, energy grid, hardware efficiency, risk tolerance, capitulation threshold, and action values.

The action values work like a simple reinforcement-learning memory. If an action improves capital, the preference for that action increases. If it performs badly, the preference decreases.

In [ ]:
class AdaptiveMiningAgent(mesa.Agent):
    """Autonomous mining entity with adaptive action preferences."""

    ACTIONS = [
        "mine", "invest", "pause", "repair", "join_pool", "leave_pool", "sell_hardware", "exit"
    ]

    def __init__(self, model: "AdaptiveMiningModel"):
        super().__init__(model)
        rng = model.rng
        p = model.params

        # Heterogeneous initial traits
        self.initial_capital = float(np.clip(rng.normal(1_000, 220), 300, 1_800))
        self.capital = self.initial_capital
        self.hash_power = float(np.clip(rng.lognormal(mean=2.2, sigma=0.45), 3, 25))
        self.hardware_efficiency = float(np.clip(rng.normal(1.0, 0.18), 0.55, 1.45))
        self.risk_tolerance = float(np.clip(rng.beta(2.2, 2.2), 0.05, 0.95))
        # Drawdown tolerance: rolling loss (over the memory window) that the
        # agent accepts before capitulating, as a fraction of initial capital.
        self.loss_tolerance = float(rng.uniform(p.loss_tolerance_min, p.loss_tolerance_max))
        self.grid = str(rng.choice(list(model.energy_grids.keys())))

        # Dynamic state
        self.status = "active"
        self.paused = False
        self.pool_id: Optional[int] = None
        self.needs_repair = False
        self.damaged_hash = 0.0
        self.last_action: Optional[str] = None
        self.previous_capital = self.capital
        self.recent_profits: List[float] = []
        self.cycles_without_profit = 0  # kept for statistics only
        self.total_rewards = 0.0
        self.total_costs = 0.0
        self.total_profit = 0.0
        self.times_won = 0
        self.times_mined = 0
        self.estimated_network_hash = None

        # Loss-streak aggregation for the event log (avoids one Lose event
        # per agent per cycle, which would dominate the process map).
        self.loss_streak_len = 0
        self.loss_streak_cost = 0.0

        # Adaptive action values. These are not fixed strategies.
        self.action_values: Dict[str, float] = {
            "mine": rng.normal(0.4, 0.2),
            "invest": rng.normal(0.15 + self.risk_tolerance, 0.25),
            "pause": rng.normal(0.05 + (1 - self.risk_tolerance) * 0.3, 0.2),
            "repair": rng.normal(0.1, 0.2),
            "join_pool": rng.normal(0.2 + (1 - self.risk_tolerance) * 0.4, 0.25),
            "leave_pool": rng.normal(0.05 + self.risk_tolerance * 0.3, 0.2),
            "sell_hardware": rng.normal(0.05 + (1 - self.risk_tolerance) * 0.2, 0.2),
            "exit": rng.normal(-0.6, 0.2),
        }

        model.log_event(self, "Initialize", reason="agent_created")

    @property
    def active(self) -> bool:
        return self.status == "active"

    def expected_cost(self) -> float:
        energy_price = self.model.energy_grids[self.grid]
        inefficiency = 1.0 / max(self.hardware_efficiency, 0.05)
        return self.hash_power * energy_price * self.model.params.cost_per_hash_factor * inefficiency

    def estimate_expected_reward(self) -> float:
        # Bounded information: agent does not know exact network hash perfectly.
        if self.estimated_network_hash is None:
            estimate = max(self.model.total_active_hash_estimate, self.hash_power)
        else:
            estimate = max(self.estimated_network_hash, self.hash_power)

        own_share = self.hash_power / estimate
        if self.pool_id is not None:
            # Pooling reduces variance, but reward is shared.
            own_share *= 0.85
        return own_share * self.model.params.block_reward_coins * self.model.coin_price

    def estimate_expected_profit(self) -> float:
        return self.estimate_expected_reward() - self.expected_cost()

    def drawdown_ratio(self) -> float:
        """Rolling loss over the recent-profit window, relative to initial capital."""
        rolling = float(np.sum(self.recent_profits)) if self.recent_profits else 0.0
        return max(0.0, -rolling) / max(self.initial_capital, 1.0)

    def exit_allowed(self) -> bool:
        """Only allow market exit when recent losses or capital pressure justify it."""
        p = self.model.params
        has_full_memory = len(self.recent_profits) >= p.max_recent_profit_memory
        sustained_negative_outlook = (
            has_full_memory
            and float(np.mean(self.recent_profits)) < 0
            and self.estimate_expected_profit() < 0
        )

        return (
            self.capital <= p.bankruptcy_threshold * 2
            or (has_full_memory and self.drawdown_ratio() >= 0.8 * self.loss_tolerance)
            or sustained_negative_outlook
        )

    def legal_actions(self) -> List[str]:
        if not self.active:
            return []
        actions = ["mine", "invest", "pause", "repair", "join_pool", "sell_hardware", "exit"]
        if self.pool_id is not None:
            actions.append("leave_pool")
            actions.remove("join_pool")
        if not self.needs_repair:
            actions.remove("repair")
        if self.capital < 60:
            actions = [a for a in actions if a in ["mine", "pause", "sell_hardware", "exit", "join_pool", "leave_pool"]]
        if not self.exit_allowed() and "exit" in actions:
            actions.remove("exit")
        return actions

    def score_actions(self) -> Dict[str, float]:
        """Combine learned action value with bounded-rational state-based pressure."""
        expected_profit = self.estimate_expected_profit()
        cash_ratio = self.capital / max(self.initial_capital, 1)
        price_momentum = self.model.price_momentum
        energy_price = self.model.energy_grids[self.grid]
        drawdown = self.drawdown_ratio()
        scores = {}

        for action in self.legal_actions():
            score = self.action_values[action]

            if action == "mine":
                score += 0.8 if expected_profit > 0 else -0.3
                score += 0.35 * price_momentum
                if self.paused:
                    score += 0.15
            elif action == "invest":
                score += 1.0 * self.risk_tolerance
                score += 0.7 if expected_profit > 0 else -0.8
                score += 0.4 * price_momentum
                score += -0.8 if cash_ratio < 0.55 else 0.0
            elif action == "pause":
                score += 0.8 if expected_profit < 0 else -0.5
                score += 0.4 if energy_price > self.model.grid_base_prices[self.grid] * 1.4 else 0.0
                score += 0.3 if cash_ratio < 0.7 else 0.0
            elif action == "repair":
                score += 1.1 if self.needs_repair else -1.0
                score += 0.3 if expected_profit > 0 else -0.1
            elif action == "join_pool":
                recent_mean = np.mean(self.recent_profits) if self.recent_profits else 0.0
                score += (1 - self.risk_tolerance) * 0.9
                score += 0.5 if recent_mean < 0 else 0.0
                score += 0.4 if self.times_mined > 4 and self.times_won == 0 else 0.0
            elif action == "leave_pool":
                score += self.risk_tolerance * 0.65
                score += 0.4 if expected_profit > 0 and price_momentum > 0 else 0.0
            elif action == "sell_hardware":
                score += 0.7 if cash_ratio < 0.45 else -0.2
                score += 0.5 if expected_profit < 0 else -0.2
            elif action == "exit":
                score += 2.2 if self.capital < self.model.params.bankruptcy_threshold * 2 else -0.7
                score += 1.1 if drawdown >= 0.8 * self.loss_tolerance else 0.0

            scores[action] = float(score)

        return scores

    def choose_action(self) -> Optional[str]:
        legal = self.legal_actions()
        if not legal:
            return None

        rng = self.model.rng
        if rng.random() < self.model.params.irrationality_rate:
            return str(rng.choice(legal))
        if rng.random() < self.model.params.exploration_rate:
            return str(rng.choice(legal))

        scores = self.score_actions()
        return softmax_choice(scores, rng, temperature=self.model.params.decision_temperature)

    def act(self) -> None:
        if not self.active:
            return

        self.previous_capital = self.capital
        action = self.choose_action()
        self.last_action = action
        if action is None:
            return

        if action == "mine":
            self.paused = False
            self.model.log_event(self, "Continue mining", reason="chosen_action")

        elif action == "invest":
            self.paused = False
            spend_fraction = 0.04 + 0.24 * self.risk_tolerance
            spend = min(self.capital * spend_fraction, self.capital - 40)
            if spend > 0:
                new_hash = (spend / self.model.params.hash_purchase_cost) * self.hardware_efficiency
                self.capital -= spend
                self.hash_power += new_hash
                self.model.log_event(self, "Invest in hash power", reason="adaptive_action", amount=spend, hash_added=new_hash)
            else:
                self.model.log_event(self, "Failed investment", reason="insufficient_capital")

        elif action == "pause":
            self.paused = True
            self.model.log_event(self, "Pause mining", reason="adaptive_action")

        elif action == "repair":
            repairable = min(self.damaged_hash, max(self.capital - 40, 0) / self.model.params.repair_cost_per_hash)
            repair_cost = repairable * self.model.params.repair_cost_per_hash
            if repairable > 0:
                self.capital -= repair_cost
                self.hash_power += repairable
                self.damaged_hash -= repairable
                self.needs_repair = self.damaged_hash > 0.1
                self.model.log_event(self, "Repair hardware", reason="adaptive_action", amount=repair_cost, hash_repaired=repairable)
            else:
                self.model.log_event(self, "Failed repair", reason="insufficient_capital")

        elif action == "join_pool":
            old_pool = self.pool_id
            self.pool_id = self.model.assign_pool(self)
            self.model.log_event(self, "Join pool", reason="variance_reduction", old_pool=old_pool, new_pool=self.pool_id)

        elif action == "leave_pool":
            old_pool = self.pool_id
            self.pool_id = None
            self.model.log_event(self, "Leave pool", reason="adaptive_action", old_pool=old_pool)

        elif action == "sell_hardware":
            sell_fraction = float(self.model.rng.uniform(0.08, 0.28))
            sold_hash = max(0.0, self.hash_power * sell_fraction)
            revenue = sold_hash * self.model.params.hash_purchase_cost * self.model.params.resale_value_fraction
            self.hash_power = max(self.model.params.min_hash_power, self.hash_power - sold_hash)
            self.capital += revenue
            self.model.log_event(self, "Sell hardware", reason="raise_cash_or_reduce_exposure", amount=revenue, hash_sold=sold_hash)

        elif action == "exit":
            self.exit_market(reason="voluntary_exit")

    def apply_hardware_failure(self) -> None:
        if not self.active:
            return
        if self.model.rng.random() < self.model.params.hardware_failure_prob:
            loss_fraction = float(self.model.rng.uniform(0.04, 0.18))
            lost_hash = self.hash_power * loss_fraction
            self.hash_power = max(self.model.params.min_hash_power, self.hash_power - lost_hash)
            self.damaged_hash += lost_hash
            self.needs_repair = True
            self.model.log_event(self, "Hardware degradation", reason="random_failure", hash_lost=lost_hash)

    def flush_loss_streak(self, reason: str = "streak_ended") -> None:
        """Log an aggregated event for a completed run of losing rounds."""
        if self.loss_streak_len > 0:
            self.model.log_event(
                self, "Losing streak", reason=reason,
                streak_length=int(self.loss_streak_len),
                streak_cost=float(self.loss_streak_cost),
            )
        self.loss_streak_len = 0
        self.loss_streak_cost = 0.0

    def update_after_profit(self, profit: float, reward: float, cost: float, won: bool) -> None:
        self.capital += reward - cost
        self.total_rewards += reward
        self.total_costs += cost
        self.total_profit += profit

        if cost > 0:
            self.times_mined += 1
        if won:
            self.times_won += 1

        self.recent_profits.append(profit)
        if len(self.recent_profits) > self.model.params.max_recent_profit_memory:
            self.recent_profits.pop(0)

        if profit <= 0:
            self.cycles_without_profit += 1
        else:
            self.cycles_without_profit = 0

        # Imperfect network estimate update.
        actual_hash = max(self.model.current_total_mining_hash, self.hash_power)
        noise = self.model.rng.normal(1.0, 0.18)
        observed = max(self.hash_power, actual_hash * noise)
        if self.estimated_network_hash is None:
            self.estimated_network_hash = observed
        else:
            self.estimated_network_hash = 0.85 * self.estimated_network_hash + 0.15 * observed

        self.learn_from_outcome(profit)

        # Exit rules: bankruptcy, or drawdown-based capitulation. The latter
        # triggers only with a full observation window, so early noise does
        # not cause immediate exits.
        if self.capital <= self.model.params.bankruptcy_threshold:
            self.exit_market(reason="bankruptcy")
        elif (
            len(self.recent_profits) >= self.model.params.max_recent_profit_memory
            and self.drawdown_ratio() >= self.loss_tolerance
        ):
            self.exit_market(reason="capitulation")

    def learn_from_outcome(self, profit: float) -> None:
        if self.last_action is None or self.last_action not in self.action_values:
            return
        scaled_reward = float(np.clip(profit / max(self.initial_capital, 1), -1.5, 1.5))
        old_value = self.action_values[self.last_action]
        lr = self.model.params.learning_rate
        self.action_values[self.last_action] = old_value + lr * (scaled_reward - old_value)

    def imitate(self) -> None:
        """Copy part of the action-value profile of a successful peer."""
        if not self.active or self.model.rng.random() >= self.model.params.imitation_rate:
            return

        candidates = [a for a in self.model.agents if a is not self and a.active and a.capital > self.capital * 1.15]
        if not candidates:
            return

        sample_size = min(10, len(candidates))
        sampled = list(self.model.rng.choice(candidates, size=sample_size, replace=False))
        mentor = max(sampled, key=lambda a: a.capital)
        blend = 0.12
        for action in self.ACTIONS:
            self.action_values[action] = (1 - blend) * self.action_values[action] + blend * mentor.action_values[action]

        self.model.log_event(self, "Imitate strategy", reason="social_learning", mentor_id=mentor.unique_id)

    def exit_market(self, reason: str) -> None:
        if not self.active:
            return
        self.flush_loss_streak(reason="exit")
        old_pool = self.pool_id
        self.status = "exited"
        self.paused = True
        self.pool_id = None
        self.model.log_event(self, "Exit market", reason=reason, old_pool=old_pool)

## 6. Adaptive mining model

The model manages the environment and interaction mechanism.

One cycle contains:

1. update coin price and energy prices;
2. apply random hardware failures;
3. let agents choose actions;
4. run a weighted-lottery mining round;
5. update learning and imitation;
6. collect model-level metrics.

In [ ]:
class AdaptiveMiningModel(mesa.Model):
    """Adaptive MAS in a stochastic cryptocurrency mining environment."""

    def __init__(self, params: MiningParams = MiningParams()):
        super().__init__(seed=params.seed)
        self.params = params
        self.rng = np.random.default_rng(params.seed)
        self.coin_price = params.initial_coin_price
        self.price_history = [self.coin_price]
        self.price_momentum = 0.0
        self.event_log: List[dict] = []
        self.model_history: List[dict] = []
        self.pool_counter = 0
        self.current_total_mining_hash = 0.0
        self.total_active_hash_estimate = 1.0
        self.event_seq = 0  # intra-cycle event sequence counter

        # Local energy grids: base prices are arbitrary but heterogeneous.
        self.grid_base_prices = {
            "Grid_A_low_stable": 0.18,
            "Grid_B_medium": 0.25,
            "Grid_C_cheap_volatile": 0.15,
            "Grid_D_high_stable": 0.36,
        }
        self.energy_grids = dict(self.grid_base_prices)
        self.grid_volatility = {
            "Grid_A_low_stable": 0.03,
            "Grid_B_medium": 0.06,
            "Grid_C_cheap_volatile": 0.12,
            "Grid_D_high_stable": 0.04,
        }

        # Create agents. Mesa automatically registers agents when super().__init__(model) is called.
        for _ in range(params.n_agents):
            AdaptiveMiningAgent(self)

        self.collect_model_metrics()

    def update_environment(self) -> None:
        p = self.params
        old_price = self.coin_price
        self.coin_price *= float(np.exp(p.price_drift + p.price_volatility * self.rng.normal()))
        self.coin_price = float(np.clip(self.coin_price, 50, 50_000))
        self.price_history.append(self.coin_price)

        if len(self.price_history) >= 6:
            self.price_momentum = (self.price_history[-1] - self.price_history[-6]) / self.price_history[-6]
        else:
            self.price_momentum = (self.coin_price - old_price) / max(old_price, 1)

        for grid, base in self.grid_base_prices.items():
            vol = self.grid_volatility[grid]
            new_price = base * float(np.exp(vol * self.rng.normal()))
            if self.rng.random() < p.energy_shock_prob:
                new_price *= float(self.rng.uniform(p.energy_shock_multiplier_min, p.energy_shock_multiplier_max))
            self.energy_grids[grid] = float(np.clip(new_price, 0.05, 2.50))

        active_hash = sum(a.hash_power for a in self.agents if a.active)
        self.total_active_hash_estimate = max(active_hash, 1.0)

    def assign_pool(self, agent: AdaptiveMiningAgent) -> int:
        """Assign agent to an existing pool or create a new pool. Pools emerge from join decisions."""
        pool_members = self.get_pool_members()
        existing_pool_ids = list(pool_members.keys())

        if len(existing_pool_ids) == 0 or (len(existing_pool_ids) < self.params.max_pools and self.rng.random() < 0.25):
            self.pool_counter += 1
            return self.pool_counter

        sample_ids = list(self.rng.choice(existing_pool_ids, size=min(5, len(existing_pool_ids)), replace=False))

        def pool_score(pool_id: int) -> float:
            members = pool_members.get(pool_id, [])
            if not members:
                return 0.0
            avg_capital = np.mean([m.capital for m in members])
            total_hash = np.sum([m.hash_power for m in members])
            return float(avg_capital + 0.5 * total_hash)

        return int(max(sample_ids, key=pool_score))

    def get_pool_members(self) -> Dict[int, List[AdaptiveMiningAgent]]:
        pools: Dict[int, List[AdaptiveMiningAgent]] = {}
        for agent in self.agents:
            if agent.active and agent.pool_id is not None:
                pools.setdefault(agent.pool_id, []).append(agent)
        return pools

    def log_event(self, agent: AdaptiveMiningAgent, activity: str, reason: str = "", **extra) -> None:
        self.event_seq += 1
        row = {
            "case_id": f"Agent_{agent.unique_id}",
            "agent_id": int(agent.unique_id),
            "cycle": int(self.steps if hasattr(self, "steps") else 0),
            "seq": int(self.event_seq),
            "activity": activity,
            "reason": reason,
            "capital": float(agent.capital),
            "hash_power": float(agent.hash_power),
            "risk_tolerance": float(agent.risk_tolerance),
            "hardware_efficiency": float(agent.hardware_efficiency),
            "grid": agent.grid,
            "energy_price": float(self.energy_grids.get(agent.grid, np.nan)),
            "coin_price": float(self.coin_price),
            "pool_id": agent.pool_id,
            "status": agent.status,
            "paused": bool(agent.paused),
            "last_action": agent.last_action,
        }
        row.update(extra)
        self.event_log.append(row)

    def step(self) -> None:
        self.event_seq = 0  # reset intra-cycle ordering
        self.update_environment()

        for agent in list(self.agents):
            agent.apply_hardware_failure()

        self.agents.shuffle_do("act")
        self.run_mining_round()
        self.agents.shuffle_do("imitate")
        self.collect_model_metrics()

    def run_mining_round(self) -> None:
        p = self.params
        miners = [a for a in self.agents if a.active and not a.paused and a.hash_power > 0]
        self.current_total_mining_hash = float(sum(a.hash_power for a in miners))

        if not miners or self.current_total_mining_hash <= 0:
            return

        pool_members = self.get_pool_members()
        lottery_entities: List[Tuple[str, object, float]] = []

        solo_miners = [a for a in miners if a.pool_id is None]
        for agent in solo_miners:
            lottery_entities.append(("solo", agent, agent.hash_power))

        for pool_id, members in pool_members.items():
            active_members = [m for m in members if m in miners]
            pool_hash = float(sum(m.hash_power for m in active_members))
            if pool_hash > 0:
                lottery_entities.append(("pool", pool_id, pool_hash))

        weights = np.array([e[2] for e in lottery_entities], dtype=float)
        probabilities = weights / weights.sum()
        winner_index = int(self.rng.choice(len(lottery_entities), p=probabilities))
        winner_type, winner_obj, _ = lottery_entities[winner_index]

        gross_reward = p.block_reward_coins * self.coin_price
        rewards = {a.unique_id: 0.0 for a in miners}
        winners = set()

        if winner_type == "solo":
            winner = winner_obj
            rewards[winner.unique_id] = gross_reward
            winners.add(winner.unique_id)
        else:
            pool_id = int(winner_obj)
            members = [m for m in pool_members.get(pool_id, []) if m in miners]
            pool_hash = sum(m.hash_power for m in members)
            distributable_reward = gross_reward * (1 - p.pool_fee)
            for member in members:
                share = member.hash_power / max(pool_hash, 1e-9)
                rewards[member.unique_id] = distributable_reward * share
                winners.add(member.unique_id)

        for agent in miners:
            cost = agent.expected_cost()
            reward = rewards.get(agent.unique_id, 0.0)
            profit = reward - cost
            won = agent.unique_id in winners

            if won:
                agent.flush_loss_streak(reason="won_round")
                self.log_event(
                    agent, "Win reward", reason="weighted_lottery",
                    reward=reward, cost=cost, profit=profit,
                )
            else:
                agent.loss_streak_len += 1
                agent.loss_streak_cost += max(0.0, -profit)

            agent.update_after_profit(profit=profit, reward=reward, cost=cost, won=won)

        for agent in [a for a in self.agents if a.active and a.paused]:
            if agent.last_action == "pause":
                pseudo_reward = -agent.estimate_expected_profit() * 0.1
                agent.learn_from_outcome(pseudo_reward)

    def finalize_run(self) -> None:
        """Flush open loss streaks and add a terminal event for survivors."""
        for agent in self.agents:
            if agent.active:
                agent.flush_loss_streak(reason="end_of_simulation")
                self.log_event(agent, "End of simulation", reason="survived")

    def collect_model_metrics(self) -> None:
        active_agents = [a for a in self.agents if a.active]
        miners = [a for a in active_agents if not a.paused]
        pools = self.get_pool_members()

        row = {
            "cycle": int(self.steps if hasattr(self, "steps") else 0),
            "coin_price": float(self.coin_price),
            "active_agents": len(active_agents),
            "mining_agents": len(miners),
            "paused_agents": len([a for a in active_agents if a.paused]),
            "exited_agents": len([a for a in self.agents if not a.active]),
            "total_hash_power": float(sum(a.hash_power for a in active_agents)),
            "mining_hash_power": float(sum(a.hash_power for a in miners)),
            "mean_capital": float(np.mean([a.capital for a in active_agents])) if active_agents else 0.0,
            "median_capital": float(np.median([a.capital for a in active_agents])) if active_agents else 0.0,
            "capital_gini": gini([a.capital for a in active_agents]) if active_agents else 0.0,
            "n_pools": len(pools),
            "agents_in_pools": len([a for a in active_agents if a.pool_id is not None]),
        }
        for grid, price in self.energy_grids.items():
            row[f"energy_{grid}"] = float(price)
        self.model_history.append(row)

    def run_model(self, n_cycles: Optional[int] = None) -> None:
        cycles = self.params.cycles if n_cycles is None else n_cycles
        for _ in range(cycles):
            if len([a for a in self.agents if a.active]) == 0:
                break
            self.step()
        self.finalize_run()

    def event_log_df(self) -> pd.DataFrame:
        return pd.DataFrame(self.event_log)

    def model_history_df(self) -> pd.DataFrame:
        return pd.DataFrame(self.model_history)

    def agent_summary_df(self) -> pd.DataFrame:
        rows = []
        for a in self.agents:
            dominant_action = max(a.action_values, key=a.action_values.get)
            rows.append({
                "agent_id": a.unique_id,
                "status": a.status,
                "capital": a.capital,
                "initial_capital": a.initial_capital,
                "capital_growth": a.capital - a.initial_capital,
                "hash_power": a.hash_power,
                "risk_tolerance": a.risk_tolerance,
                "hardware_efficiency": a.hardware_efficiency,
                "grid": a.grid,
                "pool_id": a.pool_id,
                "times_mined": a.times_mined,
                "times_won": a.times_won,
                "win_rate": a.times_won / a.times_mined if a.times_mined else 0.0,
                "total_rewards": a.total_rewards,
                "total_costs": a.total_costs,
                "total_profit": a.total_profit,
                "loss_tolerance": a.loss_tolerance,
                "dominant_learned_action": dominant_action,
                **{f"Q_{k}": v for k, v in a.action_values.items()},
            })
        return pd.DataFrame(rows)

## 7. Run the model

This section initializes the adaptive multi-agent simulation with 1,000 agents over 365 cycles. The model executes each simulation step, updating agent states, market prices, and mining pool memberships, before exporting the resulting trajectories and event logs into pandas DataFrames for analysis.

In [ ]:
# Configure simulation parameters (agent count, cycles, market volatility, etc.)
params = MiningParams(
    n_agents=1000,
    cycles=365,
    seed=101,
    price_volatility = 0.60 / np.sqrt(365),
    exploration_rate=0.09,
    imitation_rate=0.035,
    hardware_failure_prob=0.015,
    energy_shock_prob=0.04,
    price_drift= np.log(2) / 365,
)

# Instantiate the adaptive model and run the simulation
model = AdaptiveMiningModel(params)
model.run_model()

# Extract simulation results into pandas DataFrames for subsequent analysis
history_df = model.model_history_df()
event_log_df = model.event_log_df()
agent_df = model.agent_summary_df()

# Display the final 5 cycles of the model-level history
history_df.tail()

## 8. Inspect the generated event log

This is the main dataset for process mining.

Recommended process mining interpretation:

- `case_id`: one agent lifecycle;
- `activity`: action/outcome event;
- `cycle`: simulated timestamp;
- additional attributes: capital, hash power, grid, coin price, pool status, reason.

In [ ]:
# Display the first 20 records of the generated event log
event_log_df.head(20)

In [ ]:
# Count the total occurrences of each mining activity across the entire simulation run
event_log_df["activity"].value_counts()

## 9. Model-level behaviour over time

These plots provide a macro-level verification of system dynamics across simulation cycles:
1. **Agent Population Dynamics**: Tracks the number of active, actively mining, paused, and pooled miners over time.
2. **Stochastic Coin Price**: Displays the market price trajectory of the mined cryptocurrency.
3. **Hash Power Evolution**: Compares total system hash capacity against actively deployed mining hash power.
4. **Wealth Concentration (Gini)**: Shows how mining rewards and costs drive economic inequality among active agents over time.

In [ ]:
# Plot agent population dynamics (active, mining, and pooled miners over time)
plt.figure(figsize=(9, 4))
plt.plot(history_df["cycle"], history_df["active_agents"], label="Active agents")
plt.plot(history_df["cycle"], history_df["mining_agents"], label="Mining agents")
plt.plot(history_df["cycle"], history_df["agents_in_pools"], label="Agents in pools")
plt.xlabel("Cycle")
plt.ylabel("Number of agents")
plt.title("Agent population dynamics")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot the stochastic cryptocurrency market price trajectory across simulation cycles
plt.figure(figsize=(9, 4))
plt.plot(history_df["cycle"], history_df["coin_price"])
#plt.yscale()
plt.xlabel("Cycle")
plt.ylabel("Coin price (log scale)")
plt.title("Stochastic coin price")
plt.tight_layout()
plt.show()




In [ ]:
# Plot the evolution of total active hash power vs actively deployed mining power
plt.figure(figsize=(9, 4))
plt.plot(history_df["cycle"], history_df["total_hash_power"], label="Total active hash power")
plt.plot(history_df["cycle"], history_df["mining_hash_power"], label="Mining hash power")
plt.xlabel("Cycle")
plt.ylabel("Hash power")
plt.title("Hash power evolution")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot wealth concentration (Gini coefficient) among active agents over time
plt.figure(figsize=(9, 4))
plt.plot(history_df["cycle"], history_df["capital_gini"])
plt.xlabel("Cycle")
plt.ylabel("Gini coefficient")
plt.title("Capital concentration among active agents")
plt.tight_layout()
plt.show()

## 10. Agent outcomes

This section analyzes the end-state distribution of surviving and exited agents. We examine:
- Summary statistics (mean/median capital, hash power, risk tolerance, and profit) grouped by final agent status.
- The distribution of dominant learned strategies adopted by agents.
- The final capital distribution comparing surviving active agents against exited agents.

In [ ]:
# Preview the first few rows of the final agent summary dataset
agent_df.head()

In [ ]:
# Aggregate mean and median financial and hardware statistics by final agent status (active vs exited)
summary_by_status = agent_df.groupby("status").agg(
    agents=("agent_id", "count"),
    mean_capital=("capital", "mean"),
    median_capital=("capital", "median"),
    mean_risk=("risk_tolerance", "mean"),
    mean_hash=("hash_power", "mean"),
    mean_profit=("total_profit", "mean"),
).reset_index()

summary_by_status

In [ ]:
# Count how many agents adopted each mining action as their dominant learned strategy
learned_actions = agent_df["dominant_learned_action"].value_counts().rename_axis("dominant_action").reset_index(name="n_agents")
learned_actions

In [ ]:
# Plot the histogram of final capital distribution across agents
plt.figure(figsize=(8, 4))
plt.hist(agent_df["capital"], bins=35)
plt.xlabel("Final capital")
plt.ylabel("Number of agents")
plt.title("Final capital distribution")
plt.tight_layout()
plt.show()

## 11. Simple process mining preparation

This creates a process mining event log where:

- one case = one agent;
- each event = one action or outcome;
- timestamp = simulated cycle, converted to an artificial timestamp.

The event log can be exported to CSV.

In [ ]:
# Process mining preparation for Disco:
# - tie-free timestamps from (cycle, seq): cycle -> days, seq -> seconds
# - episode-based case notion: a new episode starts at each pool join/leave,
#   so traces stay short and variant analysis remains usable
# - plain column names (for standard import mapping in process mining tools like Disco)

pm_log = event_log_df.copy()
pm_log = pm_log.sort_values(["agent_id", "cycle", "seq"]).reset_index(drop=True)

pm_log["timestamp"] = (
    pd.Timestamp("2026-01-01")
    + pd.to_timedelta(pm_log["cycle"], unit="D")
    + pd.to_timedelta(pm_log["seq"], unit="s")
)

EPISODE_BOUNDARIES = {"Join pool", "Leave pool"}

pm_log["episode"] = 0
for agent_id, group in pm_log.groupby("agent_id"):
    # The boundary event itself becomes the first event of the new episode.
    pm_log.loc[group.index, "episode"] = group["activity"].isin(EPISODE_BOUNDARIES).cumsum()

pm_log["case_id"] = (
    "Agent_" + pm_log["agent_id"].astype(str)
    + "_ep" + pm_log["episode"].astype(str).str.zfill(2)
)

pm_log[["case_id", "activity", "timestamp", "capital", "hash_power", "grid", "pool_id", "reason", "status"]].head(20)

In [ ]:
import os

# Create structured output subdirectories
os.makedirs("output_mas/data", exist_ok=True)
os.makedirs("output_mas/figures", exist_ok=True)

# Export simulation data for process mining and analysis
pm_log.to_csv("output_mas/data/adaptive_crypto_mining_event_log.csv", index=False)
history_df.to_csv("output_mas/data/adaptive_crypto_mining_model_history.csv", index=False)
agent_df.to_csv("output_mas/data/adaptive_crypto_mining_agent_summary.csv", index=False)

print("Exported simulation data to output_mas/data/:")
print("- adaptive_crypto_mining_event_log.csv")
print("- adaptive_crypto_mining_model_history.csv")
print("- adaptive_crypto_mining_agent_summary.csv")

## 12. Process Mining

This section applies process mining techniques to the generated event log using the `pm4py` library. By treating individual autonomous agents as **cases**, actions as **activities**, and simulation cycles as **timestamps**, we can discover emergent behavioral workflows, analyze bottlenecks, and compare strategies across agent cohorts.

In [ ]:
# Check if Graphviz 'dot' executable is installed in PATH (required for process model rendering)
import shutil

print(shutil.which("dot"))

### 12.1 Format and clean event log for pm4py

The raw pandas DataFrame is formatted into a standard event log compatible with `pm4py`. It also cleans the data by filtering out low-value, repetitive activities like `"Continue mining"` to reduce visual noise, and prints the resulting reduction in log size.

In [ ]:
import pm4py

# 1. Format the dataframe to be compatible with pm4py
pm4py_log = pm4py.format_dataframe(
    pm_log, 
    case_id='case_id', 
    activity_key='activity', 
    timestamp_key='timestamp'
)

# 2. Filter out spam/low-value activities (such as 'Continue mining') to reduce visual noise
activities_to_remove = ["Continue mining"]
clean_log = pm4py.filter_event_attribute_values(
    pm4py_log, 
    "concept:name", 
    activities_to_remove, 
    level="event", 
    retain=False
)

print(f"Original log size: {len(pm4py_log)} events")
print(f"Cleaned log size: {len(clean_log)} events")

### 12.2 Directly-Follows Graph (DFG) discovery (Top 10 variants)

This section filters the log down to the top ten most frequent behavioral variants to establish a baseline. It then discovers and displays two Directly-Follows Graphs (DFGs): one showing the frequency of transitions between activities, and another showing the average performance time between them.

In [ ]:
import pm4py

# Keep the top 10 most frequent agent behaviors
filtered_log = pm4py.filter_variants_top_k(clean_log, 20)

# 1. Discover and view Frequency DFG 
dfg, start_activities, end_activities = pm4py.discover_dfg(filtered_log)
print("Displaying Top-K Filtered Frequency DFG...")
pm4py.view_dfg(dfg, start_activities, end_activities)

### 12.3 Petri Net discovery via Inductive Miner (Happy Path)

By isolating the data to the top five most common sequences, this analysis identifies the simulation's "happy path." It uses the Inductive Miner algorithm to discover and render a structurally sound Petri Net, representing the baseline expectation for rational agent behavior.

In [ ]:
# 1. Filter down to the absolute most common sequences (Top 5 variants)
happy_path_log = pm4py.filter_variants_top_k(clean_log, 5) 

# 2. Discover a Petri Net using the Inductive Miner on ONLY the happy path
net, initial_marking, final_marking = pm4py.discover_petri_net_inductive(happy_path_log)

# 3. View the structurally sound, highly readable Petri Net
print("Displaying 'Happy Path' Petri Net...")
pm4py.view_petri_net(net, initial_marking, final_marking)

### 12.4 BPMN Model discovery (Happy Path)

Using the exact same isolated "happy path" data from the previous step, this analysis discovers and renders a Business Process Model and Notation (BPMN) diagram. This provides a more standardized, business-friendly visualization of the core process compared to a strict Petri Net.

In [ ]:
# Discover a BPMN model using the same isolated happy path data
bpmn_model = pm4py.discover_bpmn_inductive(happy_path_log)

# View the BPMN model
print("Displaying 'Happy Path' BPMN Model...")
pm4py.view_bpmn(bpmn_model)

### 12.5 Comparative DFG analysis: Survivors vs. Losers

This section splits the log into two distinct cohorts—"Losers" and "Survivors"—based on their final outcome. By applying custom frequency thresholds to trim away rare edge cases, it renders clean, comparative DFGs to visually highlight the differing behavioral paths between failed and successful agents.

In [ ]:
from collections import Counter

# 1. Split the log based on the final outcome of the case
losers_log = pm4py.filter_end_activities(clean_log, ["Exit market"])
survivors_log = pm4py.filter_end_activities(clean_log, ["End of simulation"])

# 2. Discover raw DFGs for both segments
dfg_losers, start_l, end_l = pm4py.discover_dfg(losers_log)
dfg_survivors, start_s, end_s = pm4py.discover_dfg(survivors_log)

# --- Cohort Diagnostics: Display sample sizes and activity distributions ---
print("--- TOP 10 PATHS (SURVIVORS) ---")
for path, count in Counter(dfg_survivors).most_common(10):
    print(f"{path[0]} -> {path[1]}: {count}")

print("\n--- TOP 10 PATHS (LOSERS) ---")
for path, count in Counter(dfg_losers).most_common(10):
    print(f"{path[0]} -> {path[1]}: {count}")
print("\n")

# --- THE FIX: MUCH LOWER THRESHOLDS ---
# Dropping this down so we catch the middle loops
THRESHOLD_LOSERS = 2 
THRESHOLD_SURVIVORS = 20

# Filter the "Losers" DFG
dfg_l_filtered = {k: v for k, v in dfg_losers.items() if v > THRESHOLD_LOSERS}
surviving_l_nodes = set([node for path in dfg_l_filtered.keys() for node in path])
start_l_filt = {k: v for k, v in start_l.items() if k in surviving_l_nodes}
end_l_filt = {k: v for k, v in end_l.items() if k in surviving_l_nodes}

# Filter the "Survivors" DFG
dfg_s_filtered = {k: v for k, v in dfg_survivors.items() if v > THRESHOLD_SURVIVORS}
surviving_s_nodes = set([node for path in dfg_s_filtered.keys() for node in path])
start_s_filt = {k: v for k, v in start_s.items() if k in surviving_s_nodes}
end_s_filt = {k: v for k, v in end_s.items() if k in surviving_s_nodes}
# ------------------------------------------

# 3. View the 'Exit Market' (Failure) graph
if len(dfg_l_filtered) == 0:
    print(f"⚠️ THRESHOLD ERROR: {THRESHOLD_LOSERS} is too high for Losers. Lower it.")
else:
    print(f"Displaying 'Exit Market' DFG (Edges > {THRESHOLD_LOSERS})...")
    pm4py.view_dfg(dfg_l_filtered, start_l_filt, end_l_filt)

# 4. View the 'End of Simulation' (Success) graph
if len(dfg_s_filtered) == 0:
    print(f"⚠️ THRESHOLD ERROR: {THRESHOLD_SURVIVORS} is too high for Survivors. Lower it.")
else:
    print(f"Displaying 'End of Simulation' DFG (Edges > {THRESHOLD_SURVIVORS})...")
    pm4py.view_dfg(dfg_s_filtered, start_s_filt, end_s_filt)

### 12.6 Token-based replay conformance checking

This analysis performs conformance checking to measure how actual agent behavior deviated from expectations. It replays the full, unfiltered simulation log against the strict "happy path" Petri Net, calculating an overall fitness score and counting exactly how many traces exhibited irrational or unexpected deviations.

In [ ]:
import pm4py

# 1. Define the "Normative Model" (e.g., the top 5 most common "happy paths")
# This represents the expected protocol of a rational miner.
normative_log = pm4py.filter_variants_top_k(clean_log, 5) 
net, im, fm = pm4py.discover_petri_net_inductive(normative_log)

# 2. Replay the FULL, chaotic simulation log against this clean model
# This tests if the emergent behavior matches the designed expectations
fitness_dict = pm4py.fitness_token_based_replay(clean_log, net, im, fm)

print("--- CONFORMANCE CHECKING RESULTS ---")
print(f"Overall Log Fitness: {fitness_dict['log_fitness']:.2%}")

# FIX APPLIED HERE: Changed :.2% to :.2f%
print(f"Percentage of Perfectly Fit Traces: {fitness_dict['perc_fit_traces']:.2f}%")

# 3. Diagnose the deviations (Where did agents act "irrationally"?)
replayed_traces = pm4py.conformance_diagnostics_token_based_replay(clean_log, net, im, fm)

deviations = 0
for trace in replayed_traces:
    if not trace['trace_is_fit']:
        deviations += 1

print(f"Total traces containing deviations: {deviations}")

### 12.7 Attribute-based analytics: Grid exit rates and pooling benefits

Finally, this section converts the event log back into a pandas DataFrame to perform traditional attribute-based analytics. It dynamically locates relevant columns to aggregate market exits by their specific energy grid, and compares the average starting capital of agents who joined pools versus those who mined solo.

In [ ]:
import pandas as pd

# Convert the event log to a DataFrame
df = pm4py.convert_to_dataframe(clean_log)

print("--- AVAILABLE COLUMNS IN EVENT LOG ---")
print([col for col in df.columns])
print("\n" + "="*40 + "\n")

# Automatically detect the correct column names by matching substrings
grid_col = next((c for c in df.columns if 'grid' in c.lower()), None)
capital_col = next((c for c in df.columns if 'capital' in c.lower()), None)

# 1. Market Exits by Energy Grid
if grid_col:
    exit_events = df[df['concept:name'] == 'Exit market']
    if not exit_events.empty:
        grid_exits = exit_events.groupby(grid_col).size()
        print("--- MARKET EXITS BY ENERGY GRID ---")
        print(grid_exits)
    else:
        print("--- MARKET EXITS BY ENERGY GRID ---")
        print("No 'Exit market' events found in this subset.")
else:
    print("Could not find an energy grid column. Check the printed list above.")

print("\n" + "="*40 + "\n")

# 2. Capital vs Pool Dynamics
if capital_col:
    pool_events = df[df['concept:name'] == 'Join pool']
    if not pool_events.empty:
        # Get unique case IDs for agents who joined pools
        pool_cases = pool_events['case:concept:name'].unique()
        
        avg_pool_capital = df[df['case:concept:name'].isin(pool_cases)][capital_col].mean()
        avg_solo_capital = df[~df['case:concept:name'].isin(pool_cases)][capital_col].mean()
        
        print("--- POOL DYNAMICS BY INITIAL CAPITAL ---")
        print(f"Avg capital of agents who joined pools: {avg_pool_capital:.2f}")
        print(f"Avg capital of strictly solo agents: {avg_solo_capital:.2f}")
    else:
        print("--- POOL DYNAMICS ---")
        print("No 'Join pool' events found in the log.")
else:
    print("Could not find a capital column. Check the printed list above.")

### 12.8 Top-performing agents analysis (Wealth quantiles)

This section filters the event log by agent case IDs to isolate the highest-wealth agents (top 1% and top 10% quantiles) and analyze their specific activity profiles and success factors.

In [ ]:
import pandas as pd
import pm4py
QUANTILE = 0.99
FILTERTRESHOLD = 0.4
# 1. Load the data
df = pd.read_csv("output_mas/data/adaptive_crypto_mining_event_log.csv")

# ---------------------------------------------------------
# STEP 1: Identify the "Best" Agents (top 10% by final capital)
# ---------------------------------------------------------
final_states = df.groupby('agent_id').last()
wealth_threshold = final_states['capital'].quantile(QUANTILE)
best_agent_ids = final_states[final_states['capital'] >= wealth_threshold].index

# ---------------------------------------------------------
# STEP 2: Filter the Event Log
# ---------------------------------------------------------
best_agents_df = df[df['agent_id'].isin(best_agent_ids)].copy()

# ---------------------------------------------------------
# STEP 3: Format for PM4Py
# ---------------------------------------------------------
best_agents_log = pm4py.format_dataframe(
    best_agents_df,
    case_id='agent_id',
    activity_key='activity',
    timestamp_key='timestamp'
)

# ---------------------------------------------------------
# STEP 4: Generate a CLEAN DFG (frequency-filtered)
# ---------------------------------------------------------
dfg, start_activities, end_activities = pm4py.discover_dfg(best_agents_log)

from pm4py.algo.filtering.dfg import dfg_filtering
from pm4py.statistics.attributes.log import get as attr_get

activities_count = attr_get.get_attribute_values(best_agents_log, 'activity')

filtered_dfg, filtered_sa, filtered_ea, filtered_ac = dfg_filtering.filter_dfg_on_activities_percentage(
    dfg, start_activities, end_activities, activities_count, FILTERTRESHOLD
)

print(f"Original DFG edges: {len(dfg)} → Filtered: {len(filtered_dfg)}")
print("\nDisplaying Tactics of the Top 10% Best Performing Agents (frequency-filtered DFG)...")
pm4py.view_dfg(filtered_dfg, filtered_sa, filtered_ea)

### 12.9 Behavioral comparison: Top 10% vs. Bottom 10% performers

Discovers and compares directly-follows graphs (DFGs) for the wealthiest 10% of agents against the poorest 10%, highlighting differences in strategic decision-making and activity sequencing.

In [ ]:
import pandas as pd
import pm4py
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pm4py.algo.filtering.dfg import dfg_filtering
from pm4py.statistics.attributes.log import get as attr_get
import os

# 1. Load data
print("Loading data...")
df = pd.read_csv("output_mas/data/adaptive_crypto_mining_event_log.csv")

# 2. Get top 10% and bottom 10% agents by final capital
final_states = df.groupby('agent_id').last()
top_thresh = final_states['capital'].quantile(0.90)
bottom_thresh = final_states['capital'].quantile(0.10)

top_ids = final_states[final_states['capital'] >= top_thresh].index
bottom_ids = final_states[final_states['capital'] <= bottom_thresh].index

top_df = df[df['agent_id'].isin(top_ids)].copy()
bottom_df = df[df['agent_id'].isin(bottom_ids)].copy()

# 3. Format as PM4Py logs
print("Formatting logs...")
top_log = pm4py.format_dataframe(top_df, case_id='agent_id', activity_key='activity', timestamp_key='timestamp')
bottom_log = pm4py.format_dataframe(bottom_df, case_id='agent_id', activity_key='activity', timestamp_key='timestamp')

# 4. Discover and filter DFGs
print("Discovering DFGs...")
top_dfg, top_sa, top_ea = pm4py.discover_dfg(top_log)
bottom_dfg, bottom_sa, bottom_ea = pm4py.discover_dfg(bottom_log)

top_ac = attr_get.get_attribute_values(top_log, 'activity')
bottom_ac = attr_get.get_attribute_values(bottom_log, 'activity')

filt_top_dfg, filt_top_sa, filt_top_ea, _ = dfg_filtering.filter_dfg_on_activities_percentage(top_dfg, top_sa, top_ea, top_ac, 0.3)
filt_bot_dfg, filt_bot_sa, filt_bot_ea, _ = dfg_filtering.filter_dfg_on_activities_percentage(bottom_dfg, bottom_sa, bottom_ea, bottom_ac, 0.3)

# 5. Save visualizations
print("Saving visualizations...")
pm4py.save_vis_dfg(filt_top_dfg, filt_top_sa, filt_top_ea, 'output_mas/figures/top_dfg.png')
pm4py.save_vis_dfg(filt_bot_dfg, filt_bot_sa, filt_bot_ea, 'output_mas/figures/bottom_dfg.png')

# 6. Display comparison
print("\n--- Statistical Comparison ---")
print(f"Top 10% Agents: {len(top_ids)} agents, Mean Final Capital: ${final_states.loc[top_ids, 'capital'].mean():.2f}, Total Events: {len(top_df)}")
print(f"Bottom 10% Agents: {len(bottom_ids)} agents, Mean Final Capital: ${final_states.loc[bottom_ids, 'capital'].mean():.2f}, Total Events: {len(bottom_df)}")

# Create plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

if os.path.exists('output_mas/figures/top_dfg.png'):
    img1 = mpimg.imread('output_mas/figures/top_dfg.png')
    ax1.imshow(img1)
ax1.set_title("Top 10% Agents Strategy (Filtered)", fontsize=16)
ax1.axis('off')

if os.path.exists('output_mas/figures/bottom_dfg.png'):
    img2 = mpimg.imread('output_mas/figures/bottom_dfg.png')
    ax2.imshow(img2)
ax2.set_title("Bottom 10% Agents Strategy (Filtered)", fontsize=16)
ax2.axis('off')

plt.tight_layout()
plt.savefig('output_mas/figures/cell1_best_vs_worst.png')
print("Saved comparison figure to output_mas/figures/cell1_best_vs_worst.png")


### 12.10 Exit market analysis: Tracing agent capitulation

Examines the behavioral traces of agents who exited the market to understand the sequence of events leading to bankruptcy or voluntary capitulation.

In [ ]:
import pandas as pd
import pm4py
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pm4py.algo.filtering.dfg import dfg_filtering
from pm4py.statistics.attributes.log import get as attr_get
import os

print("Loading data...")
df = pd.read_csv("output_mas/data/adaptive_crypto_mining_event_log.csv")

# 1. Find agents who exited
exit_agents = df[df['activity'] == 'Exit market']['agent_id'].unique()
print(f"Found {len(exit_agents)} agents who exited.")

# 2. Extract the last 5 activities before 'Exit market'
death_traces = []
for aid in exit_agents:
    agent_df = df[df['agent_id'] == aid]
    agent_df_no_exit = agent_df[agent_df['activity'] != 'Exit market']
    death_traces.append(agent_df_no_exit.tail(5))

death_df = pd.concat(death_traces)

# 3. Discover AND FILTER death trace DFG
death_log = pm4py.format_dataframe(death_df, case_id='agent_id', activity_key='activity', timestamp_key='timestamp')
dfg, sa, ea = pm4py.discover_dfg(death_log)

# ---> NEW: Filter out the noise to fix spaghetti <---
activities_count = attr_get.get_attribute_values(death_log, 'activity')
# Keep top 40% most frequent edges (tune this down to 0.2 if still messy)
filt_dfg, filt_sa, filt_ea, _ = dfg_filtering.filter_dfg_on_activities_percentage(dfg, sa, ea, activities_count, 0.5)

# 4. Print stats
exit_events = df[df['activity'] == 'Exit market']
print("\n--- Survival Analysis ---")
print(f"Total Agents Dead: {len(exit_agents)}")
print(f"Mean capital at death: ${exit_events['capital'].mean():.2f}")
print(f"Mean cycle of death: {exit_events['cycle'].mean():.1f}")
print("\nExit reasons:")
print(exit_events['reason'].value_counts())

print("\nMost common activities right before exit:")
last_activities = death_df.groupby('agent_id').last()['activity']
print(last_activities.value_counts().head(5))

# 5. Display DFG
pm4py.save_vis_dfg(filt_dfg, filt_sa, filt_ea, 'output_mas/figures/survival_dfg.png')
if os.path.exists('output_mas/figures/survival_dfg.png'):
    plt.figure(figsize=(15, 10))
    plt.imshow(mpimg.imread('output_mas/figures/survival_dfg.png'))
    plt.title("The Death Spiral: Last 5 Activities Before Death (Filtered)", fontsize=16)
    plt.axis('off')
    plt.show()

### 12.11 Tiered wealth distribution and activity profiles

Categorizes agents into wealth quartiles and analyzes how activity frequencies (such as pooling, pausing, and investing) vary across performance tiers.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

print("Loading data...")
df = pd.read_csv("output_mas/data/adaptive_crypto_mining_event_log.csv")

# 1. Group agents into tiers
final_states = df.groupby('agent_id').last()

top_thresh = final_states['capital'].quantile(0.90)
bottom_thresh = final_states['capital'].quantile(0.10)
mid_upper = final_states['capital'].quantile(0.75)
mid_lower = final_states['capital'].quantile(0.25)

top_ids = final_states[final_states['capital'] >= top_thresh].index
bottom_ids = final_states[final_states['capital'] <= bottom_thresh].index
mid_ids = final_states[(final_states['capital'] >= mid_lower) & (final_states['capital'] <= mid_upper)].index

# 2. Get capital per cycle per agent
print("Processing trajectories...")
# To get the trajectory, we want the LAST event in each cycle for each agent
cycle_states = df.groupby(['cycle', 'agent_id']).last().reset_index()

# 3. Compute mean and std per tier per cycle
top_traj = cycle_states[cycle_states['agent_id'].isin(top_ids)].groupby('cycle')['capital'].agg(['mean', 'std']).fillna(0)
mid_traj = cycle_states[cycle_states['agent_id'].isin(mid_ids)].groupby('cycle')['capital'].agg(['mean', 'std']).fillna(0)
bot_traj = cycle_states[cycle_states['agent_id'].isin(bottom_ids)].groupby('cycle')['capital'].agg(['mean', 'std']).fillna(0)

# 4. Plot
print("Plotting...")
plt.style.use('default') # Avoid seaborn dependency error if not installed
plt.figure(figsize=(12, 7))

# Top 10%
plt.plot(top_traj.index, top_traj['mean'], color='green', linewidth=2, label='Top 10% Agents')
plt.fill_between(top_traj.index, top_traj['mean'] - top_traj['std'], top_traj['mean'] + top_traj['std'], color='green', alpha=0.1)

# Middle 50%
plt.plot(mid_traj.index, mid_traj['mean'], color='blue', linewidth=2, label='Middle 50% Agents')
plt.fill_between(mid_traj.index, mid_traj['mean'] - mid_traj['std'], mid_traj['mean'] + mid_traj['std'], color='blue', alpha=0.1)

# Bottom 10%
plt.plot(bot_traj.index, bot_traj['mean'], color='red', linewidth=2, label='Bottom 10% Agents')
plt.fill_between(bot_traj.index, bot_traj['mean'] - bot_traj['std'], bot_traj['mean'] + bot_traj['std'], color='red', alpha=0.1)

plt.title('Capital Trajectory: Top vs Middle vs Bottom Agents', fontsize=16)
plt.xlabel('Cycle', fontsize=14)
plt.ylabel('Mean Capital ($)', fontsize=14)
plt.yscale('log') # Log scale because of exponential growth
plt.legend(fontsize=12)
plt.grid(True, which="both", ls="-", alpha=0.2)
plt.tight_layout()

plt.savefig('output_mas/figures/cell3_capital.png')
print("Saved figure to output_mas/figures/cell3_capital.png")


### 12.12 Social learning and mentor imitation network

Investigates the social learning mechanism by tracking which successful mentors were most frequently imitated by peers, and how imitation correlated with final wealth accumulation.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

print("Loading data...")
df = pd.read_csv("output_mas/data/adaptive_crypto_mining_event_log.csv")

# 1. Filter to Imitate strategy
imitations = df[df['activity'] == 'Imitate strategy'].dropna(subset=['mentor_id'])

# 2. Count how many times each mentor was imitated
mentor_counts = imitations['mentor_id'].value_counts()

# 3. Get final capital for each agent
final_states = df.groupby('agent_id').last()

# 4. Find the top 10 most-imitated agents
top_mentors = mentor_counts.head(10)

top_mentor_ids = top_mentors.index.astype(int)
top_mentor_counts = top_mentors.values
top_mentor_capitals = final_states.loc[top_mentor_ids, 'capital'].values

print("\n--- Social Network Analysis ---")
print("Top 10 Most Imitated Agents:")
print(pd.DataFrame({
    'Agent ID': top_mentor_ids,
    'Times Imitated': top_mentor_counts,
    'Final Capital ($)': [f"{c:.2f}" for c in top_mentor_capitals]
}))

# 5. Visualization
plt.style.use('default')
plt.figure(figsize=(12, 6))

# Create a colormap based on capital
norm = plt.Normalize(top_mentor_capitals.min(), top_mentor_capitals.max())
colors = cm.viridis(norm(top_mentor_capitals))

bars = plt.bar([str(i) for i in top_mentor_ids], top_mentor_counts, color=colors)

# Add colorbar
sm = plt.cm.ScalarMappable(cmap=cm.viridis, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=plt.gca())
cbar.set_label('Final Capital ($)', fontsize=12)

plt.title('Top 10 Most Imitated Agents (Influencers)', fontsize=16)
plt.xlabel('Agent ID', fontsize=14)
plt.ylabel('Times Imitated', fontsize=14)
plt.tight_layout()

plt.savefig('output_mas/figures/cell4_social.png')
print("Saved figure to output_mas/figures/cell4_social.png")


### 12.13 Longitudinal activity patterns of top-performing agents

Analyzes how the action choices of the top 10% wealthiest agents evolved across simulation cycles over time.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

print("Loading data...")
df = pd.read_csv("output_mas/data/adaptive_crypto_mining_event_log.csv")

# 1. Get top 10% agents
final_states = df.groupby('agent_id').last()
top_thresh = final_states['capital'].quantile(0.90)
top_ids = final_states[final_states['capital'] >= top_thresh].index

top_df = df[df['agent_id'].isin(top_ids)]

# 2. Main activities to focus on
main_activities = [
    'Continue mining', 'Invest in hash power', 'Sell hardware', 
    'Join pool', 'Leave pool', 'Pause mining'
]

# 3. Calculate percentage distribution per grid
grid_types = sorted(top_df['grid'].unique())
heatmap_data = []

print("\n--- Grid Strategy Analysis (Top 10% Agents) ---")
for act in main_activities:
    row = []
    for grid in grid_types:
        grid_df = top_df[top_df['grid'] == grid]
        total = len(grid_df)
        if total > 0:
            count = len(grid_df[grid_df['activity'] == act])
            pct = (count / total) * 100
        else:
            pct = 0
        row.append(pct)
    heatmap_data.append(row)

heatmap_data = np.array(heatmap_data)

# 4. Visualization
plt.style.use('default')
fig, ax = plt.subplots(figsize=(10, 6))

im = ax.imshow(heatmap_data, cmap='YlGnBu')

# We want to show all ticks...
ax.set_xticks(np.arange(len(grid_types)))
ax.set_yticks(np.arange(len(main_activities)))
# ... and label them with the respective list entries
ax.set_xticklabels([g.replace('Grid_', '') for g in grid_types])
ax.set_yticklabels(main_activities)

# Rotate the tick labels and set their alignment.
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

# Loop over data dimensions and create text annotations.
for i in range(len(main_activities)):
    for j in range(len(grid_types)):
        text = ax.text(j, i, f"{heatmap_data[i, j]:.1f}%",
                       ha="center", va="center", color="black" if heatmap_data[i, j] < 20 else "white")

ax.set_title("Activity Distribution by Energy Grid (Top 10% Agents)", fontsize=16)
fig.tight_layout()

plt.colorbar(im, label='Percentage of Events (%)')
plt.savefig('output_mas/figures/cell5_grid.png')
print("Saved figure to output_mas/figures/cell5_grid.png")


### 12.14 Agent trait correlations and financial outcomes

Visualizes relationships between initial agent traits (such as risk tolerance, hardware efficiency, and initial capital) and their final financial performance.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
try:
    import seaborn as sns
    HAS_SNS = True
except ModuleNotFoundError:
    HAS_SNS = False
    
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Loading data...")
df = pd.read_csv("output_mas/data/adaptive_crypto_mining_event_log.csv")

# 1. Get next activity for each row
print("Computing next activities...")
df['next_activity'] = df.groupby('agent_id')['activity'].shift(-1)

# 2. Filter to rows where current activity is 'Continue mining'
mining_df = df[df['activity'] == 'Continue mining'].dropna(subset=['next_activity'])

# 3. Get top 5 next activities
top_next = mining_df['next_activity'].value_counts().head(5).index
decision_df = mining_df[mining_df['next_activity'].isin(top_next)]

print("\n--- Decision Point Analysis ---")
print("Top next activities after 'Continue mining':")
print(mining_df['next_activity'].value_counts().head(5))

# 4. Summary table
print("\nMean Market Conditions at Decision Point:")
summary = decision_df.groupby('next_activity')[['coin_price', 'energy_price', 'capital']].mean()
print(summary)

# 5. Visualization
plt.style.use('default')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

try:
    sns.boxplot(data=decision_df, x='next_activity', y='coin_price', ax=ax1)
    sns.boxplot(data=decision_df, x='next_activity', y='energy_price', ax=ax2)
except Exception:
    # Fallback if seaborn fails
    decision_df.boxplot(column='coin_price', by='next_activity', ax=ax1, grid=False)
    decision_df.boxplot(column='energy_price', by='next_activity', ax=ax2, grid=False)

ax1.set_title('Coin Price at Decision Point', fontsize=14)
ax1.set_xlabel('Next Activity', fontsize=12)
ax1.set_ylabel('Coin Price', fontsize=12)
ax1.tick_params(axis='x', rotation=45)

ax2.set_title('Energy Price at Decision Point', fontsize=14)
ax2.set_xlabel('Next Activity', fontsize=12)
ax2.set_ylabel('Energy Price', fontsize=12)
ax2.tick_params(axis='x', rotation=45)

fig.suptitle("What Drives the Next Action After 'Continue Mining'?", fontsize=16)
plt.tight_layout()

plt.savefig('output_mas/figures/cell6_decision.png')
print("Saved figure to output_mas/figures/cell6_decision.png")


### 12.15 Energy grid paradox analysis

Evaluates the distribution of top-performing agents across different local energy grids to investigate whether low energy costs guarantee mining success, or if volatility and shocks outweigh base price advantages.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

print("Loading data...")
df = pd.read_csv("output_mas/data/adaptive_crypto_mining_event_log.csv")

# 1. Grid Paradox Data
final_states = df.groupby('agent_id').last()
top_10 = final_states[final_states['capital'] >= final_states['capital'].quantile(0.90)]
grid_counts = top_10['grid'].value_counts()

# Clean up grid names for display
grid_labels = [g.replace('Grid_', '') for g in grid_counts.index]
grid_sizes = grid_counts.values
colors = ['#ff9999','#66b3ff','#99ff99','#ffcc99']

# 2. Strategic Pooling Data (Leave -> Join -> Invest sequence)
df['next_act'] = df.groupby('agent_id')['activity'].shift(-1)
df['next_next_act'] = df.groupby('agent_id')['activity'].shift(-2)

strat_hop = df[(df['activity'] == 'Leave pool') & 
               (df['next_act'] == 'Join pool') & 
               (df['next_next_act'] == 'Invest in hash power')]

top_10_ids = top_10.index
top_strat_avg = len(strat_hop[strat_hop['agent_id'].isin(top_10_ids)]) / len(top_10_ids)
bot_strat_avg = len(strat_hop[~strat_hop['agent_id'].isin(top_10_ids)]) / (len(final_states) - len(top_10_ids))

# 3. Visualization
plt.style.use('default')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: Pie Chart (The Grid Paradox)
wedges, texts, autotexts = ax1.pie(grid_sizes, colors=colors, labels=grid_labels, autopct='%1.1f%%', 
                                   startangle=90, pctdistance=0.85, 
                                   wedgeprops=dict(width=0.3, edgecolor='w'))
ax1.set_title('The Grid Paradox:\nTop 10% Agents by Starting Grid', fontsize=16, pad=20)
plt.setp(autotexts, size=12, weight="bold")
plt.setp(texts, size=12)

# Right: Bar Chart (Strategic Pooling)
bars = ax2.bar(['Top 10% Agents\n(Winners)', 'Bottom 90% Agents\n(Losers)'], 
               [top_strat_avg, bot_strat_avg], 
               color=['green', 'red'], alpha=0.7)

ax2.set_title('Strategic Pooling:\nAvg. "Leave → Join → Invest" Sequences', fontsize=16, pad=20)
ax2.set_ylabel('Avg. Executions per Agent', fontsize=14)
ax2.tick_params(axis='x', labelsize=12)

# Add values on bars
for bar in bars:
    yval = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, yval + 0.05, f'{yval:.2f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 13. Replicating the winning strategy (Generalisation test)

This section extracts the learned action preferences of the baseline top-10% agents (their averaged Q-values) and transplants them into fresh simulation runs across 10 new random seeds. We check:
1. **Process Conformance**: Do they reproduce the discovered happy path protocol? (Measured via token-based replay fitness).
2. **Financial Performance**: Does the learned strategy yield higher survival rates, average capital, and lower inequality (Gini) across unseen market conditions?

In [ ]:
from dataclasses import replace
import numpy as np
import pandas as pd
import pm4py


def winning_profile_from_run(agent_summary: pd.DataFrame, top_quantile: float = 0.90):
    q_cols = [c for c in agent_summary.columns if c.startswith("Q_")]
    if not q_cols:
        raise ValueError("No Q_ columns — run `agent_df = model.agent_summary_df()` first.")
    thresh = agent_summary["capital"].quantile(top_quantile)
    top = agent_summary[agent_summary["capital"] >= thresh]
    profile = {c[2:]: float(top[c].mean()) for c in q_cols}   # 'Q_mine' -> 'mine'
    return profile, len(top)


def prepare_clean_log(event_df: pd.DataFrame, remove=("Continue mining",)):
    log = event_df.copy().sort_values(["agent_id", "cycle", "seq"]).reset_index(drop=True)
    log["timestamp"] = (
        pd.Timestamp("2026-01-01")
        + pd.to_timedelta(log["cycle"], unit="D")
        + pd.to_timedelta(log["seq"], unit="s")
    )
    boundaries = {"Join pool", "Leave pool"}
    log["episode"] = 0
    for _, g in log.groupby("agent_id"):
        log.loc[g.index, "episode"] = g["activity"].isin(boundaries).cumsum()
    log["case_id"] = (
        "Agent_" + log["agent_id"].astype(str)
        + "_ep" + log["episode"].astype(str).str.zfill(2)
    )
    fmt = pm4py.format_dataframe(log, case_id="case_id", activity_key="activity", timestamp_key="timestamp")
    return pm4py.filter_event_attribute_values(fmt, "concept:name", list(remove), level="event", retain=False)


def run_with_profile(base_params, profile, seed, jitter=0.0):
    m = AdaptiveMiningModel(replace(base_params, seed=seed))
    jr = np.random.default_rng(seed + 9999) if jitter > 0 else None
    for ag in m.agents:
        ag.action_values = {
            a: profile.get(a, ag.action_values[a]) + (float(jr.normal(0, jitter)) if jr is not None else 0.0)
            for a in ag.action_values
        }
    m.run_model()
    return m


def performance_summary(agent_summary: pd.DataFrame, label: str):
    n = len(agent_summary)
    return {
        "run": label,
        "survival_%": 100.0 * (agent_summary["status"] == "active").sum() / n,
        "mean_capital": float(agent_summary["capital"].mean()),
        "median_capital": float(agent_summary["capital"].median()),
        "gini": float(gini(agent_summary["capital"].values)),
        "mean_profit": float(agent_summary["total_profit"].mean()),
    }


# Guard: ensure agent_df has the learned Q_ columns
if "agent_df" not in globals() or not any(c.startswith("Q_") for c in agent_df.columns):
    agent_df = model.agent_summary_df()

# 1. Normative net from the BASELINE happy path (top-5 variants)
baseline_clean = prepare_clean_log(event_log_df)
normative_log = pm4py.filter_variants_top_k(baseline_clean, 5)
net, im, fm = pm4py.discover_petri_net_inductive(normative_log)

# 2. Winning profile from the baseline top 10%
profile, n_top = winning_profile_from_run(agent_df, 0.90)
print(f"Winning profile from {n_top} top-10% agents:")
for a, v in profile.items():
    print(f"  Q_{a:<13} {v:+.3f}")

# 3. Baseline reference row
rows = []
base_fit = pm4py.fitness_token_based_replay(baseline_clean, net, im, fm)
base_row = performance_summary(agent_df, "baseline (seed 101)")
base_row["log_fitness_%"] = 100.0 * base_fit["log_fitness"]
base_row["fit_traces_%"] = base_fit["perc_fit_traces"]
rows.append(base_row)

# 4. Seeded replications on 10 NEW seeds
NEW_SEEDS = [202, 303, 404, 505, 606, 707, 808, 909, 1010, 1111]
for s in NEW_SEEDS:
    print(f"Running seeded replication: seed {s} ...")
    m = run_with_profile(params, profile, seed=s)
    clean_s = prepare_clean_log(m.event_log_df())
    fit = pm4py.fitness_token_based_replay(clean_s, net, im, fm)
    row = performance_summary(m.agent_summary_df(), f"seeded (seed {s})")
    row["log_fitness_%"] = 100.0 * fit["log_fitness"]
    row["fit_traces_%"] = fit["perc_fit_traces"]
    rows.append(row)

comparison = pd.DataFrame(rows).set_index("run")
print("\n--- BASELINE vs SEEDED-WINNING-STRATEGY (per seed) ---")
comparison.round(2)

### 13.1 Aggregate performance over seeded runs

Aggregates key evaluation metrics across all 10 seeded runs and computes the mean and standard deviation compared against the baseline simulation.

In [ ]:
# Aggregate metrics across the 10 seeded runs and plot performance comparison against baseline
import matplotlib.pyplot as plt

seeded = comparison.loc[[r for r in comparison.index if r.startswith("seeded")]]
base = comparison.loc["baseline (seed 101)"]

metrics = ["survival_%", "log_fitness_%", "mean_capital", "median_capital", "gini"]

agg = pd.DataFrame({
    "baseline": base[metrics],
    "seeded_mean": seeded[metrics].mean(),
    "seeded_std": seeded[metrics].std(),
    "seeded_min": seeded[metrics].min(),
    "seeded_max": seeded[metrics].max(),
})
agg["delta_vs_baseline"] = agg["seeded_mean"] - agg["baseline"]
print(f"Aggregated over {len(seeded)} seeded runs:\n")
print(agg.round(2))

# Visual: seeded mean bar (±1 std) vs baseline dashed line
fig, axes = plt.subplots(1, len(metrics), figsize=(18, 4))
for ax, m in zip(axes, metrics):
    ax.bar(["seeded"], [agg.loc[m, "seeded_mean"]],
           yerr=[agg.loc[m, "seeded_std"]], capsize=6, color="green", alpha=0.7)
    ax.axhline(agg.loc[m, "baseline"], color="gray", ls="--", lw=1.5, label="baseline")
    ax.set_title(m)
    ax.legend(fontsize=8)
fig.suptitle("Seeded winning strategy over 10 seeds (mean ± std vs baseline)", fontsize=14)
plt.tight_layout()
plt.show()